# Lab 3: Nim with Minimax and Alpha-Beta, plus a full EDA of the game data
Environment (Anaconda): `conda activate ailab`, then `jupyter notebook`. Run cells top to bottom with **Shift+Enter**.

**Game (Nim):** piles of stones; players A (MAX) and B (MIN) take turns; each turn you take 1 or more stones from ONE pile; whoever takes the last stone wins. Value: +1 A wins, -1 B wins.

**Part 1 (cells 1 to 4):** the game and the search. **Part 2 (cells 5 to 8):** Step 3 on real data: build a table of games, check quality, describe it, ask questions, plot, interpret.

## Cell 1: The game

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

START = (3, 4, 5)            # stones in each pile: the board of the game (Nim)
nodes = 0                    # how many positions the search looks at

def moves(piles, order="small_first"):
    """All positions we can reach by taking some stones from ONE pile."""
    result = []
    for i, n in enumerate(piles):
        takes = range(1, n + 1) if order == "small_first" else range(n, 0, -1)
        for take in takes:
            new = list(piles)
            new[i] = n - take
            result.append(tuple(new))
    return result

print(moves((1, 2)))

ModuleNotFoundError: No module named 'numpy'

## Cell 2: Minimax

In [ ]:
def minimax(piles, player):
    """+1 if player A wins, -1 if player B wins (both play perfectly)."""
    global nodes
    nodes += 1
    if sum(piles) == 0:                     # the previous player took the last stone and won
        return -1 if player == "A" else 1
    values = [minimax(nxt, "B" if player == "A" else "A") for nxt in moves(piles)]
    return max(values) if player == "A" else min(values)

## Cell 3: Alpha-beta pruning

In [ ]:
def alphabeta(piles, player, alpha, beta, order):
    global nodes
    nodes += 1
    if sum(piles) == 0:
        return -1 if player == "A" else 1
    if player == "A":
        value = -2
        for nxt in moves(piles, order):
            value = max(value, alphabeta(nxt, "B", alpha, beta, order))
            alpha = max(alpha, value)
            if alpha >= beta:
                break                      # prune: B will never allow this branch
        return value
    value = 2
    for nxt in moves(piles, order):
        value = min(value, alphabeta(nxt, "A", alpha, beta, order))
        beta = min(beta, value)
        if alpha >= beta:
            break                          # prune: A already has a better option
    return value

## Cell 4: Count the nodes

In [ ]:
nodes = 0
v1 = minimax(START, "A")
n_minimax = nodes

nodes = 0
v2 = alphabeta(START, "A", -2, 2, "small_first")
n_ab = nodes

nodes = 0
v3 = alphabeta(START, "A", -2, 2, "big_first")
n_good = nodes

print("value of START (+1 = A wins):", v1, v2, v3)
print("minimax nodes:           ", n_minimax)
print("alpha-beta (natural):    ", n_ab)
print("alpha-beta (big first):  ", n_good)

## Cell 5: Build the data table (200 games) and save it as CSV

In [ ]:
def best_move(piles, player):
    best_nxt, best_value = None, None
    for nxt in moves(piles, "big_first"):
        value = alphabeta(nxt, "B" if player == "A" else "A", -2, 2, "big_first")
        better = best_value is None or (value > best_value if player == "A" else value < best_value)
        if better:
            best_nxt, best_value = nxt, value
    return best_nxt

rng = np.random.default_rng(0)
rows = []
for game in range(200):
    piles = tuple(int(x) for x in rng.integers(1, 5, size=3))     # random start: 3 piles of 1..4 stones
    nodes = 0
    start_value = alphabeta(piles, "A", -2, 2, "big_first")
    nodes_ab = nodes
    nodes = 0
    minimax(piles, "A")
    nodes_mm = nodes
    board, player, n_moves = piles, "A", 0            # play: A is smart, B plays at random
    while sum(board) > 0:
        if player == "A":
            board = best_move(board, "A")
        else:
            options = moves(board)
            board = options[rng.integers(len(options))]
        player = "B" if player == "A" else "A"
        n_moves += 1
    rows.append({"p1": piles[0], "p2": piles[1], "p3": piles[2], "total": sum(piles),
                 "nim_sum": piles[0] ^ piles[1] ^ piles[2], "first_wins": start_value > 0,
                 "nodes_minimax": nodes_mm, "nodes_alphabeta": nodes_ab, "moves": n_moves,
                 "winner": "B" if player == "A" else "A"})

df = pd.DataFrame(rows)
df.to_csv("nim_games.csv", index=False)               # keep the data in a file (Step 2)
print(df.head())

## Cell 6: Data quality and descriptive statistics

In [ ]:
print(df.shape)                                       # rows, columns
print("missing values:", df.isna().sum().sum())
print("duplicated rows:", df.duplicated().sum())
print(df.dtypes)
print(df[["total", "nodes_minimax", "nodes_alphabeta", "moves"]].describe().round(1))
x = df["nodes_minimax"]
print("mean", round(x.mean(), 1), "| variance", round(x.var(), 1), "| std", round(x.std(), 1), "| median", x.median())

## Cell 7: Analytic questions

In [ ]:
print("1) A wins from", round(df["first_wins"].mean() * 100), "% of the start positions")
print("2) nim_sum != 0 agrees with first_wins:", ((df["nim_sum"] != 0) == df["first_wins"]).mean())
print("3) alpha-beta uses", round(df["nodes_alphabeta"].sum() / df["nodes_minimax"].sum() * 100, 1), "% of the minimax nodes")
print("4) smart A wins", round((df["winner"] == "A").mean() * 100), "% of the games against a random B")
print(df[["total", "nodes_minimax", "nodes_alphabeta"]].corr().round(2))

## Cell 8: Charts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 6))
df["nodes_minimax"].plot(kind="hist", bins=20, ax=axes[0, 0], title="Histogram: minimax nodes")
df.groupby("total")["first_wins"].mean().plot(kind="bar", ax=axes[0, 1], title="A win rate by total stones")
df[["nodes_minimax", "nodes_alphabeta"]].plot(kind="box", logy=True, ax=axes[1, 0], title="Box plot (log scale)")
axes[1, 1].scatter(df["total"], df["nodes_minimax"], s=8)
axes[1, 1].set_yscale("log")
axes[1, 1].set_title("Scatter: total stones vs nodes")
plt.tight_layout()
plt.show()

## Interpretation (write it yourself in README)
Use the pattern: **observation, reason, conclusion**. Example: *A wins from 88 percent of the starts (observation); most starts have nim_sum different from 0 (reason); so the first player has an advantage (conclusion).*

# Part 3: Motion output

## Cell 9: Animation of one game

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

rng = np.random.default_rng(1)
board, player = START, "A"
history = [(board, "start")]
while sum(board) > 0:
    old = board
    if player == "A":
        board = best_move(board, "A")
    else:
        options = moves(board)
        board = options[rng.integers(len(options))]
    i = [k for k in range(3) if board[k] != old[k]][0]
    history.append((board, f"{player} takes {old[i] - board[i]} from pile {i}"))
    player = "B" if player == "A" else "A"
winner = "B" if player == "A" else "A"

fig, ax = plt.subplots(figsize=(4, 3), dpi=70)

def draw(t):
    piles, text = history[min(t, len(history) - 1)]
    ax.clear()
    ax.bar(["pile 0", "pile 1", "pile 2"], piles, color=["#4f46e5", "#ec4899", "#f59e0b"])
    ax.set_ylim(0, max(START) + 1)
    ax.set_title(text + ("   winner: " + winner if t >= len(history) - 1 else ""), fontsize=9)

anim = FuncAnimation(fig, draw, frames=len(history) + 3, interval=800)
plt.close(fig)
HTML(anim.to_jshtml())

## Cell 10: Interactive graphical Nim game (Human vs Minimax AI)

Run this cell in Jupyter. It draws the piles as stones and provides buttons so you can choose a pile and remove stones. The AI responds using `best_move`. Requires `ipywidgets` (install with `pip install ipywidgets` if needed).

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

# Start a fresh human-vs-AI game.
game_state = {"board": START, "player": "Human", "over": False}

pile_picker = widgets.Dropdown(
    options=[("Pile 1", 0), ("Pile 2", 1), ("Pile 3", 2)],
    value=0, description="Choose pile:"
)
take_picker = widgets.Dropdown(
    options=[("1 stone", 1)], value=1, description="Remove:"
)
take_button = widgets.Button(description="Take stones", button_style="success")
reset_button = widgets.Button(description="New game", button_style="")
status = widgets.HTML()
game_output = widgets.Output()

def refresh_take_options(*args):
    board = game_state["board"]
    pile_index = pile_picker.value
    available = board[pile_index]
    options = [(f"{n} stone" if n == 1 else f"{n} stones", n)
               for n in range(1, available + 1)]
    take_picker.options = options
    if options:
        take_picker.value = options[0][1]
    take_button.disabled = (game_state["over"] or available == 0)

def draw_nim_board():
    board = game_state["board"]
    with game_output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7, 3))
        colors = ["#4f46e5", "#ec4899", "#f59e0b"]
        for i, count in enumerate(board):
            if count:
                ax.scatter([i] * count, list(range(1, count + 1)),
                           s=650, color=colors[i], edgecolor="white", linewidth=1.5)
            ax.text(i, 0, f"Pile {i+1}: {count}", ha="center", va="top",
                    fontsize=11, fontweight="bold")
        ax.set_xlim(-0.6, 2.6)
        ax.set_ylim(-0.8, max(START) + 1.2)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title("Nim — Human vs Minimax AI", fontsize=14)
        for spine in ax.spines.values():
            spine.set_visible(False)
        plt.show()
        plt.close(fig)

def update_status(message=None):
    if message:
        status.value = f"<b>{message}</b>"
    elif game_state["over"]:
        status.value = "<b>Game over.</b>"
    else:
        status.value = "<b>Your turn:</b> choose a non-empty pile and how many stones to remove."

def ai_turn():
    board = game_state["board"]
    if sum(board) == 0:
        game_state["over"] = True
        return "You took the last stone. You win!"
    next_board = best_move(board, "A")
    game_state["board"] = next_board
    if sum(next_board) == 0:
        game_state["over"] = True
        return "The AI took the last stone. AI wins."
    return "AI made its move. Your turn."

def on_take_clicked(_):
    if game_state["over"]:
        return
    board = list(game_state["board"])
    pile_index, amount = pile_picker.value, take_picker.value
    if board[pile_index] < amount or amount < 1:
        update_status("Invalid move — choose a non-empty pile.")
        return
    board[pile_index] -= amount
    game_state["board"] = tuple(board)
    if sum(board) == 0:
        game_state["over"] = True
        message = "You took the last stone. You win!"
    else:
        message = ai_turn()
    refresh_take_options()
    draw_nim_board()
    update_status(message)
    take_button.disabled = game_state["over"]

def on_reset_clicked(_):
    game_state.update({"board": START, "player": "Human", "over": False})
    refresh_take_options()
    draw_nim_board()
    update_status("New game started. Your turn.")
    take_button.disabled = False

pile_picker.observe(refresh_take_options, names="value")
take_button.on_click(on_take_clicked)
reset_button.on_click(on_reset_clicked)

refresh_take_options()
draw_nim_board()
update_status()
display(widgets.VBox([
    widgets.HBox([pile_picker, take_picker, take_button, reset_button]),
    status,
    game_output
]))

## Challenges
Smart B too, 4 piles, other pile ranges, ordering by nim-sum, normal versus misere rule.